In [6]:
from dotenv import load_dotenv
load_dotenv()  # 自动寻找 .env 文件并加载到 os.environ

True

## **配置GPU加速处理**

In [7]:
import os
os.environ["TORCH_DEVICE"] = "cuda"
os.environ["INFERENCE_RAM"] = "6"  # 你的显卡是 6GB
os.environ["LOWRES_IMAGE_DPI"] = "256"

## 运行marker进行pdf解析

In [8]:
from marker.converters.pdf import PdfConverter
from marker.models import create_model_dict
from marker.output import text_from_rendered
from marker.config.parser import ConfigParser


# ------------------ 配置 ------------------
input_pdf_path = r"knowledgeBase/pdfParsed_input/《营造法式》解读 第2版术语库.pdf"

config = {
    "output_format": "markdown",
    "paginate_output": True,
    "force_ocr":True,
    # "lowres_image_dpi":256,
    "TORCH_DEVICE": "cuda",  # 明确指定
    "inference_ram": 6,     # 明确指定显存
    "batch_multiplier": 1,  
    "page_range":"14",
    # "debug":True,
}
model_dict = create_model_dict()
config_parser = ConfigParser(config)

converter = PdfConverter(
    config = config_parser.generate_config_dict(),
    artifact_dict = model_dict,
    llm_service=config_parser.get_llm_service()

)

rendered = converter(input_pdf_path)
text, _, images = text_from_rendered(rendered)
print("转换完成")


Detecting bboxes: 100%|██████████| 1/1 [00:00<00:00,  1.46it/s]
Detecting bboxes: 0it [00:00, ?it/s]

转换完成


In [9]:
print(text)



{14}------------------------------------------------




## 保存本次pdf解析结果

In [3]:
import os
from datetime import datetime

output_dir = r"knowledgeBase\pdfParsed_output"
os.makedirs(output_dir, exist_ok=True)

# ===== 2. 获取输入文件名（不含后缀）=====
input_filename = os.path.splitext(os.path.basename(input_pdf_path))[0]
# ===== 3. 生成时间戳，避免覆盖 =====
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
# ===== 4. 保存 Markdown 结果 =====
text_output_path = os.path.join(
    output_dir,
    f"{input_filename}_result_{timestamp}.md"
)

with open(text_output_path, "w", encoding="utf-8") as f:
    f.write(text)

# ===== 5. 保存图片（如果有）=====
image_output_dir = os.path.join(
    output_dir,
    f"{input_filename}_images_{timestamp}"
)
saved_images = []

if images:
    os.makedirs(image_output_dir, exist_ok=True)
    for idx, (image_name, image_data) in enumerate(images.items()):
        image_path = os.path.join(image_output_dir, image_name)
        # image_data 可能是 PIL Image 或 bytes
        if hasattr(image_data, "save"):  # PIL Image
            image_data.save(image_path)
        else:  # bytes
            with open(image_path, "wb") as img_f:
                img_f.write(image_data)
        saved_images.append(image_path)

# ===== 6. Notebook 输出说明 =====
print("✅ PDF 解析结果已保存")
print(f"📄 文本/markdown 文件: {text_output_path}")

if saved_images:
    print(f"图片目录: {image_output_dir}")
    print(f"图片数量: {len(saved_images)}")
else:
    print("本次解析未产生可导出的图片")


✅ PDF 解析结果已保存
📄 文本/markdown 文件: knowledgeBase\pdfParsed_output\《营造法式》解读 第2版术语库_result_20260201_225221.md
本次解析未产生可导出的图片


## 解析pdf的书签数据作为精确目录

In [ ]:
import fitz  # PyMuPDF
import os

def export_pdf_bookmarks(pdf_path):
    # 1. 检查文件是否存在
    if not os.path.exists(pdf_path):
        print(f"错误：找不到文件 '{pdf_path}'，请检查路径。")
        return

    try:
        # 2. 打开 PDF
        doc = fitz.open(pdf_path)
        toc = doc.get_toc()  # 获取原始书签数据列表
        
        if not toc:
            print("提示：该 PDF 文件内部没有书签数据（逻辑目录为空）。")
            return

        # 3. 准备输出文件名
        output_filename = os.path.splitext(pdf_path)[0] + "_目录.txt"
        
        print(f"\n正在提取: {pdf_path}")
        print("-" * 60)
        
        with open(output_filename, "w", encoding="utf-8") as f:
            # 写入标题行
            header = f"{'层级':<5} | {'页码':<5} | {'章节标题'}"
            f.write(header + "\n" + "="*60 + "\n")
            print(header)
            print("-" * 60)

            for entry in toc:
                level, title, page = entry[0], entry[1], entry[2]
                
                # 构建缩进和视觉引导符
                indent = "    " * (level - 1)
                visual_prefix = "|-- " if level > 1 else ""
                
                # 格式化单行内容
                display_line = f"L{level:<4} | P{page:<4} | {indent}{visual_prefix}{title}"
                file_line = f"[{level}] 页码:{page:<4} | {indent}{visual_prefix}{title}"
                
                # 逻辑 A：打印到控制台
                print(display_line)
                
                # 逻辑 B：输出到本地文件
                f.write(file_line + "\n")

        print("-" * 60)
        print(f"✅ 导出成功！本地文件已生成：\n👉 {os.path.abspath(output_filename)}")

    except Exception as e:
        print(f"运行出错：{e}")
    finally:
        if 'doc' in locals():
            doc.close()

# --- 执行部分 ---
# 替换为你电脑上的 PDF 绝对路径或相对路径
pdf_file = r"knowledgeBase\pdfParsed_input\《营造法式》解读 第2版 (潘谷西；何建中著) (Z-Library).pdf" 
export_pdf_bookmarks(pdf_file)